<a href="https://colab.research.google.com/github/Ayu-sshhhhh/MindBloom/blob/main/emotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
from tensorflow.keras.models import model_from_json
import numpy as np

In [ ]:
import tensorflow as tf
import os

try:
    model = tf.keras.models.model_from_json(
        open("facialemotionmodel.json").read(),
        custom_objects={"Sequential": tf.keras.Sequential}
    )
    model.load_weights("facialemotionmodel.h5")
    print("Model loaded successfully!")
except OSError as e:
    if "truncated file" in str(e):
        print("Error: The 'facialemotionmodel.h5' file appears to be corrupted or incomplete.")
        print("Please ensure 'facialemotionmodel.h5' is correctly uploaded and not truncated.")
        print("You can try re-uploading the file to resolve this issue.")
    else:
        print(f"An OSError occurred: {e}")
except FileNotFoundError:
    print("Error: One or both model files ('facialemotionmodel.json', 'facialemotionmodel.h5') were not found.")
    print("Please ensure they are uploaded to the Colab environment.")
except Exception as e:
    print(f"An unexpected error occurred while loading the model: {e}")

Model loaded successfully!


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.load_weights("facialemotionmodel.h5")
haar_file=cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade=cv2.CascadeClassifier(haar_file)

In [ ]:
def extract_features(image):
    feature = np.array(image)
    feature = feature.reshape(1,48,48,1)
    return feature/255.0


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from IPython.display import Javascript
display(Javascript('''
  navigator.mediaDevices.getUserMedia({video: true})
    .then(stream => {
      console.log("Camera permission granted!");
      stream.getTracks().forEach(t => t.stop());
    })
    .catch(err => console.error("Camera denied:", err));
'''))

<IPython.core.display.Javascript object>

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

def take_photo():
    js_code = '''
        async function takePhoto() {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);
            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();
            await new Promise((resolve) => capture.onclick = resolve);
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(track => track.stop());
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        takePhoto()
    '''
    data = eval_js(js_code)
    binary = b64decode(data.split(',')[1])
    img_array = np.frombuffer(binary, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    return img

# Take a photo and run emotion detection on it
im = take_photo()
gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, 1.3, 5)
# ... rest of your detection code
webcam = cv2.VideoCapture(1)
labels = {0 : 'angry', 1 : 'disgust', 2 : 'fear', 3 : 'happy', 4 : 'neutral', 5 : 'sad', 6 : 'surprise'}
while True:
    i, im = webcam.read()
    if not i:
        print("Failed to read from camera")
        break
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    try:
        for (p,q,r,s) in faces:
            image = gray[q:q+s,p:p+r]
            cv2.rectangle(im,(p,q),(p+r,q+s),(255,0,0),2)
            image = cv2.resize(image,(48,48))
            img = extract_features(image)
            pred = model.predict(img)
            prediction_label = labels[pred.argmax()]
            # print("Predicted Output:", prediction_label)
            # cv2.putText(im,prediction_label)
            cv2.putText(im, '% s' %(prediction_label), (p-10, q-10),cv2.FONT_HERSHEY_COMPLEX_SMALL,2, (0,0,255))
        cv2.imshow("Output",im)
        cv2.waitKey(27)
    except cv2.error:
        pass

Failed to read from camera


In [ ]:
webcam = cv2.VideoCapture(1)
labels = {0 : 'angry', 1 : 'disgust', 2 : 'fear', 3 : 'happy', 4 : 'neutral', 5 : 'sad', 6 : 'surprise'}
while True:
    i, im = webcam.read()
    if not i:
        print("Failed to read from camera")
        break
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    try:
        for (p,q,r,s) in faces:
            image = gray[q:q+s,p:p+r]
            cv2.rectangle(im,(p,q),(p+r,q+s),(255,0,0),2)
            image = cv2.resize(image,(48,48))
            img = extract_features(image)
            pred = model.predict(img)
            prediction_label = labels[pred.argmax()]
            # print("Predicted Output:", prediction_label)
            # cv2.putText(im,prediction_label)
            cv2.putText(im, '% s' %(prediction_label), (p-10, q-10),cv2.FONT_HERSHEY_COMPLEX_SMALL,2, (0,0,255))
        cv2.imshow("Output",im)
        cv2.waitKey(27)
    except cv2.error:
        pass

Failed to read from camera


In [ ]:
from google.colab.patches import cv2_imshow
import cv2

# Upload an image instead of using webcam
from google.colab import files
uploaded = files.upload()  # select an image from your PC

import numpy as np
from PIL import Image
import io

fname = list(uploaded.keys())[0]
img = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_COLOR)

# Now run emotion detection on uploaded image
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, 1.3, 5)

Saving facialemotionmodel.h5 to facialemotionmodel (1).h5


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import base64
import numpy as np
import cv2

# ── Optimized: predict every 5 frames only ──────────────────────
PREDICT_EVERY = 5  # reduce lag
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
emotion_emoji  = {'angry':'😠','disgust':'🤢','fear':'😨','happy':'😊','neutral':'😐','sad':'😢','surprise':'😲'}

def extract_features_single(gray_face):
    """Proper preprocessing matching training pipeline"""
    img = cv2.resize(gray_face, (48, 48))
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=-1)   # (48,48,1)
    img = np.expand_dims(img, axis=0)    # (1,48,48,1)
    return img

# JS video streamer
display(Javascript('''
    window._stopStream = false;
    window._pendingResolve = null;
    window._stream = null;
    window._video = null;
    window._canvas = null;
    window._label = null;
    window._container = null;

    async function initCam() {
        if (window._container) return;
        window._container = document.createElement('div');
        window._container.style.cssText = 'border:2px solid #4a7c59;border-radius:12px;padding:10px;max-width:640px;background:#0a1309';
        document.body.appendChild(window._container);

        window._label = document.createElement('div');
        window._label.style.cssText = 'color:#7aab8a;font-family:sans-serif;font-size:18px;font-weight:bold;padding:6px;text-align:center';
        window._label.innerText = '🔍 Initializing...';
        window._container.appendChild(window._label);

        window._video = document.createElement('video');
        window._video.style.cssText = 'width:100%;border-radius:8px;transform:scaleX(-1)';
        window._video.autoplay = true;
        window._container.appendChild(window._video);

        const info = document.createElement('div');
        info.style.cssText = 'color:#5a7a62;font-size:12px;text-align:center;margin-top:6px;font-family:sans-serif';
        info.innerText = 'Press Escape to stop';
        window._container.appendChild(info);

        window._canvas = document.createElement('canvas');
        window._canvas.width = 640;
        window._canvas.height = 480;

        window._stream = await navigator.mediaDevices.getUserMedia({
            video: { width: 640, height: 480, facingMode: 'user', frameRate: 30 }
        });
        window._video.srcObject = window._stream;
        await window._video.play();

        document.addEventListener('keydown', e => {
            if (e.key === 'Escape') window._stopStream = true;
        });

        requestAnimationFrame(function loop() {
            if (!window._stopStream) requestAnimationFrame(loop);
            if (window._pendingResolve) {
                if (window._stopStream) {
                    window._pendingResolve('');
                } else {
                    window._canvas.getContext('2d').drawImage(
                        window._video, 0, 0, 640, 480);
                    window._pendingResolve(
                        window._canvas.toDataURL('image/jpeg', 0.6));
                }
                window._pendingResolve = null;
            }
        });
    }

    async function getFrame(label) {
        await initCam();
        if (label) window._label.innerText = label;
        if (window._stopStream) return '';
        return new Promise(r => { window._pendingResolve = r; });
    }
'''))

# ── Main detection loop ─────────────────────────────────────────
frame_count   = 0
last_emotion  = 'neutral'
last_emoji    = '😐'
last_conf     = 0
label_display = '🔍 Starting...'

print("📷 Camera starting — press Escape to stop")

while True:
    js_data = eval_js(f'getFrame("{label_display}")')
    if not js_data:
        print("⛔ Stopped.")
        break

    # Decode frame
    img_bytes  = b64decode(js_data.split(',')[1])
    img_array  = np.frombuffer(img_bytes, dtype=np.uint8)
    frame      = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    if frame is None:
        continue

    frame_count += 1

    # Only predict every N frames to reduce lag
    if frame_count % PREDICT_EVERY == 0:
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Improve face detection with better params
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30),
            flags=cv2.CASCADE_SCALE_IMAGE
        )

        if len(faces) > 0:
            # Use largest face
            faces = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)
            (p, q, r, s) = faces[0]

            # Preprocess correctly
            face_gray = gray[q:q+s, p:p+r]
            img_input = extract_features_single(face_gray)

            pred = model.predict(img_input, verbose=0)[0]
            idx  = pred.argmax()

            last_emotion  = emotion_labels[idx]
            last_emoji    = emotion_emoji[last_emotion]
            last_conf     = int(pred[idx] * 100)
            label_display = f'{last_emoji} {last_emotion.upper()} — {last_conf}%'
        else:
            label_display = '👤 No face detected'

<IPython.core.display.Javascript object>

📷 Camera starting — press Escape to stop


KeyboardInterrupt: 

In [ ]:
!pip install -q opencv-python-headless

import cv2
import numpy as np
from keras.models import model_from_json
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import base64

print('✅ All imports done')


# ── Cell 2: Upload your model files ────────────────────────────
from google.colab import files

print('📁 Upload facialemotionmodel.json and facialemotionmodel.h5')
uploaded = files.upload()
print('✅ Uploaded:', list(uploaded.keys()))


# ── Cell 3: Load your trained model ────────────────────────────
import tensorflow as tf

# Load model architecture
json_file = open('facialemotionmodel.json', 'r')
model_json = json_file.read()
json_file.close()

model = model_from_json(model_json, custom_objects={'Sequential': tf.keras.Sequential})
model.load_weights('facialemotionmodel.h5')

print('✅ Model loaded successfully!')
model.summary()

# Emotion labels — must match training order
LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
EMOJI  = {'angry':'😠', 'disgust':'🤢', 'fear':'😨', 'happy':'😊',
           'neutral':'😐', 'sad':'😢', 'surprise':'😲'}

# Load Haar cascade face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print('✅ Face detector ready!')


# ── Cell 4: Live Camera Emotion Detection ──────────────────────
# Press Escape or click Stop (■) in Colab to end the stream

def preprocess_face(gray_face):
    """Preprocess face exactly like training pipeline"""
    face = cv2.resize(gray_face, (48, 48))
    face = face.astype('float32') / 255.0
    face = np.reshape(face, (1, 48, 48, 1))
    return face

def draw_overlay(frame, x, y, w, h, emotion, confidence):
    """Draw styled bounding box and label"""
    # Green bounding box
    cv2.rectangle(frame, (x, y), (x+w, y+h), (74, 124, 89), 2)

    # Background for label
    label_text = f'{emotion.upper()}  {confidence}%'
    (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (x, y-35), (x+tw+10, y), (74, 124, 89), -1)
    cv2.putText(frame, label_text, (x+5, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return frame

# JavaScript for camera stream
display(Javascript('''
    window._mbStop = false;
    window._mbResolve = null;
    window._mbInited = false;

    async function mbInit() {
        if (window._mbInited) return;
        window._mbInited = true;

        // Container
        const wrap = document.createElement('div');
        wrap.style.cssText = 'max-width:660px;margin:10px 0;font-family:sans-serif';
        document.body.appendChild(wrap);

        // Title
        const title = document.createElement('div');
        title.style.cssText = 'background:#2d5a3d;color:#fff;padding:8px 14px;border-radius:8px 8px 0 0;font-size:15px;font-weight:600';
        title.innerHTML = '🎭 MindBloom — Live Emotion Detection';
        wrap.appendChild(title);

        // Status bar
        window._mbStatus = document.createElement('div');
        window._mbStatus.style.cssText = 'background:#162019;color:#7aab8a;padding:6px 14px;font-size:13px;min-height:28px';
        window._mbStatus.innerText = '🔍 Starting camera...';
        wrap.appendChild(window._mbStatus);

        // Video
        window._mbVideo = document.createElement('video');
        window._mbVideo.style.cssText = 'width:100%;display:block;transform:scaleX(-1);background:#000';
        window._mbVideo.autoplay = true;
        wrap.appendChild(window._mbVideo);

        // Processed frame display
        window._mbImg = document.createElement('img');
        window._mbImg.style.cssText = 'width:100%;display:none;';
        wrap.appendChild(window._mbImg);

        // Footer
        const footer = document.createElement('div');
        footer.style.cssText = 'background:#0a1309;color:#5a7a62;padding:6px 14px;border-radius:0 0 8px 8px;font-size:11px';
        footer.innerText = 'Press Escape or Colab Stop button (■) to end session';
        wrap.appendChild(footer);

        // Canvas for frame capture
        window._mbCanvas = document.createElement('canvas');
        window._mbCanvas.width  = 640;
        window._mbCanvas.height = 480;

        // Start webcam
        try {
            const stream = await navigator.mediaDevices.getUserMedia({
                video: { width:640, height:480, facingMode:'user', frameRate:{ideal:30} }
            });
            window._mbVideo.srcObject = stream;
            await window._mbVideo.play();
            window._mbStatus.innerText = '✅ Camera ready — detecting emotions...';
        } catch(e) {
            window._mbStatus.innerText = '❌ Camera error: ' + e.message;
            window._mbStop = true;
        }

        // Escape key to stop
        document.addEventListener('keydown', e => {
            if (e.key === 'Escape') { window._mbStop = true; }
        });

        // Frame loop
        function loop() {
            if (!window._mbStop) requestAnimationFrame(loop);
            if (window._mbResolve) {
                if (window._mbStop) {
                    window._mbResolve('');
                } else {
                    window._mbCanvas.getContext('2d')
                        .drawImage(window._mbVideo, 0, 0, 640, 480);
                    window._mbResolve(
                        window._mbCanvas.toDataURL('image/jpeg', 0.65));
                }
                window._mbResolve = null;
            }
        }
        requestAnimationFrame(loop);
    }

    async function mbGetFrame(statusText, processedFrame) {
        await mbInit();
        if (statusText)  window._mbStatus.innerText = statusText;
        if (processedFrame) {
            window._mbVideo.style.display = 'none';
            window._mbImg.style.display   = 'block';
            window._mbImg.src = processedFrame;
        }
        if (window._mbStop) return '';
        return new Promise(r => { window._mbResolve = r; });
    }
'''))

print('🎥 Camera UI loaded. Starting detection loop...')
print('⏹  Press Escape or Colab Stop button to stop')

# ── Detection loop ──────────────────────────────────────────────
PREDICT_EVERY = 3      # predict every N frames (lower = more accurate, higher = faster)
frame_count   = 0
last_label    = '🔍 Detecting...'
processed_b64 = ''

while True:
    # Get raw frame from browser
    js_reply = eval_js(f'mbGetFrame("{last_label}", "{processed_b64}")')
    if not js_reply:
        print('⛔ Stream ended.')
        break

    # Decode JPEG frame
    img_bytes = b64decode(js_reply.split(',')[1])
    arr       = np.frombuffer(img_bytes, dtype=np.uint8)
    frame     = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if frame is None:
        continue

    frame_count += 1

    # Mirror frame (matches video mirror transform)
    frame = cv2.flip(frame, 1)

    if frame_count % PREDICT_EVERY == 0:
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detect faces
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        if len(faces) > 0:
            # Use the largest detected face
            faces_sorted = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)
            (x, y, w, h) = faces_sorted[0]

            # Preprocess exactly like training
            face_roi   = gray[y:y+h, x:x+w]
            face_input = preprocess_face(face_roi)

            # Predict
            preds      = model.predict(face_input, verbose=0)[0]
            idx        = preds.argmax()
            emotion    = LABELS[idx]
            confidence = int(preds[idx] * 100)
            emoji      = EMOJI[emotion]

            # Draw overlay on frame
            frame = draw_overlay(frame, x, y, w, h, emotion, confidence)
            last_label = f'{emoji} {emotion.upper()}  —  {confidence}% confidence'
        else:
            last_label = '👤 No face detected — look at camera'

    # Encode processed frame and send back to browser
    _, buf      = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
    processed_b64 = 'data:image/jpeg;base64,' + base64.b64encode(buf).decode()

print('✅ Done!')

✅ All imports done
📁 Upload facialemotionmodel.json and facialemotionmodel.h5


Saving facialemotionmodel.h5 to facialemotionmodel (2).h5
Saving facialemotionmodel.json to facialemotionmodel (2).json
✅ Uploaded: ['facialemotionmodel (2).h5', 'facialemotionmodel (2).json']
✅ Model loaded successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 128)    │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 23, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 10, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,232,199 (16.14 MB)

...

 Non-trainable params: 0 (0.00 B)

✅ Face detector ready!


<IPython.core.display.Javascript object>

🎥 Camera UI loaded. Starting detection loop...
⏹  Press Escape or Colab Stop button to stop


KeyboardInterrupt: 